# CornerDetector — EfficientNetV2-S + U-Net Egitimi

**Adimlar:**
1. GPU kontrolu
2. Google Drive baglama (checkpoint kaydetmek icin)
3. Repo klonlama + bagimliliklar
4. Egitim (~1-2 saat T4'te)
5. Modeli Drive'a kaydet

---
Baslamadan once: **Runtime > Change runtime type > T4 GPU** secin.

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('GPU YOK — Runtime > Change runtime type > T4 GPU seciniz!')

In [ ]:
# 2. Google Drive bagla
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/CornerDetector_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint dizini:', CHECKPOINT_DIR)

In [ ]:
# 3. Repo klonla / guncelle
import os
if not os.path.exists('/content/checkerboard-corner-detector'):
    !git clone https://github.com/HeliumNitrate/checkerboard-corner-detector.git /content/checkerboard-corner-detector
else:
    !git -C /content/checkerboard-corner-detector pull
    print('Repo guncellendi.')

In [ ]:
# 4. Bagimliliklar
!pip install timm>=0.9 --quiet
import timm
print('timm:', timm.__version__)

In [ ]:
# 5. Modeli test et (pretrained ImageNet)
import sys, os

if not os.path.exists('/content/checkerboard-corner-detector'):
    os.system('git clone https://github.com/HeliumNitrate/checkerboard-corner-detector.git /content/checkerboard-corner-detector')
    print('Repo klonlandi.')

CNN_DIR = '/content/checkerboard-corner-detector'
os.chdir(CNN_DIR)
if CNN_DIR not in sys.path:
    sys.path.insert(0, CNN_DIR)

from model.detector import CornerDetector
import torch

m = CornerDetector(pretrained=True)
n = sum(p.numel() for p in m.parameters())
x = torch.zeros(1, 3, 256, 256).cuda() if torch.cuda.is_available() else torch.zeros(1, 3, 256, 256)
m = m.cuda() if torch.cuda.is_available() else m
out = m(x)
print(f'Params : {n/1e6:.2f}M')
print(f'Output : {list(out.shape)}  (beklenen: [1, 1, 256, 256])')
print('Model OK!')

In [ ]:
# 6. EGITIM
# - Drive'da best.pt varsa KALDIGI YERDEN devam eder
# - Yoksa bastan baslar (ImageNet pretrained)
# - Epoch basi ~690s (T4, n_train=2000, workers=0)
# - 40 epoch toplam ~7.5 saat

import sys, os, argparse

if not os.path.exists('/content/checkerboard-corner-detector'):
    os.system('git clone https://github.com/HeliumNitrate/checkerboard-corner-detector.git /content/checkerboard-corner-detector')

CNN_DIR = '/content/checkerboard-corner-detector'
os.chdir(CNN_DIR)
if CNN_DIR not in sys.path:
    sys.path.insert(0, CNN_DIR)

# Mevcut checkpoint varsa bildir
best_path = CHECKPOINT_DIR + '/best.pt'
if os.path.exists(best_path):
    import torch
    ckpt = torch.load(best_path, map_location='cpu')
    print(f'Checkpoint bulundu: epoch {ckpt["epoch"]}, val_loss {ckpt["val_loss"]:.4f}')
    print('Egitim bu noktadan devam edecek.')
else:
    print('Checkpoint yok — bastan baslanacak.')

from train import train

args = argparse.Namespace(
    epochs     = 40,
    batch      = 16,
    n_train    = 2000,
    n_val      = 1000,
    img_size   = 256,
    lr         = 1e-4,
    workers    = 0,
    save       = best_path,
    pretrained = True,
    resume     = True,
)

train(args)

In [ ]:
# 7. Checkpoint kontrol
import os, torch

best_path = CHECKPOINT_DIR + '/best.pt'
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location='cpu')
    print('Checkpoint OK!')
    print('  En iyi epoch :', ckpt['epoch'])
    print('  Val loss     :', round(ckpt['val_loss'], 4))
    print('  Konum        :', best_path)
else:
    print('Checkpoint bulunamadi:', best_path)

In [ ]:
# 8. Gorsel test
import sys, os

if not os.path.exists('/content/checkerboard-corner-detector'):
    os.system('git clone https://github.com/HeliumNitrate/checkerboard-corner-detector.git /content/checkerboard-corner-detector')

CNN_DIR = '/content/checkerboard-corner-detector'
os.chdir(CNN_DIR)
if CNN_DIR not in sys.path:
    sys.path.insert(0, CNN_DIR)

import torch, numpy as np
import matplotlib.pyplot as plt
from model.detector import CornerDetector
from data.synthesize import generate_sample
from infer import predict_heatmap, heatmap_to_corners

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(CHECKPOINT_DIR + '/best.pt', map_location=device)
model  = CornerDetector(pretrained=False).to(device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

rng    = np.random.default_rng(99)
sample = generate_sample(img_size=256, rng=rng)
img    = sample['image']
gt_corners = sample['corners'].reshape(-1, 2)
gt_valid   = gt_corners[~np.isnan(gt_corners[:, 0])]

heatmap      = predict_heatmap(model, img, device=device)
pred_corners = heatmap_to_corners(heatmap, min_distance=8, threshold=0.3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Distorted image'); axes[0].axis('off')
axes[1].imshow(heatmap, cmap='hot'); axes[1].set_title('Predicted heatmap'); axes[1].axis('off')
axes[2].imshow(img, cmap='gray')
axes[2].scatter(gt_valid[:, 1],     gt_valid[:, 0],     s=8,  c='lime', label=f'GT ({len(gt_valid)})')
axes[2].scatter(pred_corners[:, 1], pred_corners[:, 0], s=12, c='red',  marker='x', label=f'Pred ({len(pred_corners)})')
axes[2].set_title('GT (yesil) vs Tahmin (kirmizi)'); axes[2].legend(fontsize=8); axes[2].axis('off')
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR + '/inference_test.png', dpi=150)
plt.show()
print('Gorsel kaydedildi:', CHECKPOINT_DIR + '/inference_test.png')